In [ ]:
# ============================================================
# MST-Transformer  on PAMAP2 dataset
# 
# ============================================================

import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, confusion_matrix, classification_report)

# ---------------------------
# Settings
# ---------------------------
DATA_PATH = "PAMAP2_Dataset"
SEQ_LEN = 100
STRIDE = 20
MAX_WORKERS = 0
BATCH_SIZE = 64
EPOCHS = 20
LR = 1e-3
MIXUP_ALPHA = 0.4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------
# 1) Load + Preprocess
# ---------------------------
def load_pamap2_data(data_path=DATA_PATH):
    print("Loading PAMAP2 dataset from:", data_path)
    all_data, all_labels = [], []
    column_names = ['timestamp','activity_id','heart_rate']
    imu_columns = [
        'temperature','acceleration_16g_x','acceleration_16g_y','acceleration_16g_z',
        'acceleration_6g_x','acceleration_6g_y','acceleration_6g_z',
        'gyroscope_x','gyroscope_y','gyroscope_z',
        'magnetometer_x','magnetometer_y','magnetometer_z',
        'orientation_1','orientation_2','orientation_3','orientation_4'
    ]
    for pos in ['hand','chest','ankle']:
        for c in imu_columns:
            column_names.append(f"{pos}_{c}")

    for subfolder in ["Protocol", "Optional"]:
        path = os.path.join(data_path, subfolder)
        if not os.path.exists(path):
            continue
        for f in os.listdir(path):
            if f.endswith(".dat"):
                fp = os.path.join(path, f)
                try:
                    df = pd.read_csv(fp, sep=' ', header=None, names=column_names)
                    df = df[df['activity_id'] != 0]
                    orientation_cols = [c for c in df.columns if 'orientation' in c]
                    df = df.drop(columns=orientation_cols)
                    feature_cols = [c for c in df.columns if c not in ['timestamp','activity_id']]
                    all_data.append(df[feature_cols])
                    all_labels.append(df['activity_id'])
                except Exception as e:
                    print("Error loading", fp, e)
    if not all_data:
        raise FileNotFoundError("No PAMAP2 .dat files found.")
    X = pd.concat(all_data, ignore_index=True)
    y = pd.concat(all_labels, ignore_index=True)
    print("Loaded:", X.shape, "labels:", y.shape)
    return X, y

def create_sequences(X, y, seq_len=SEQ_LEN, stride=STRIDE):
    X_seq, y_seq = [], []
    for i in range(0, len(X) - seq_len, stride):
        X_seq.append(X[i:i+seq_len])
        lab = y[i:i+seq_len]
        y_seq.append(np.argmax(np.bincount(lab)))
    return np.array(X_seq, np.float32), np.array(y_seq, np.int64)

print("1) Loading and preprocessing...")
X_df, y_ser = load_pamap2_data(DATA_PATH)
X_df = X_df.fillna(X_df.mean())
const_cols = X_df.columns[X_df.nunique() <= 1]
if len(const_cols) > 0:
    X_df = X_df.drop(columns=const_cols)

X = X_df.values.astype(np.float32)
y = y_ser.values
le = LabelEncoder()
y_enc = le.fit_transform(y)
class_names = list(le.classes_)
n_classes = len(class_names)
scaler = StandardScaler()
X = scaler.fit_transform(X)
X_seq, y_seq = create_sequences(X, y_enc)
print("Sequences:", X_seq.shape, "Labels:", np.bincount(y_seq))
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_seq, y_seq, test_size=0.3, stratify=y_seq, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

# ---------------------------
# Dataset + Loaders
# ---------------------------
class SeqDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = torch.utils.data.DataLoader(SeqDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = torch.utils.data.DataLoader(SeqDataset(X_val, y_val), batch_size=BATCH_SIZE)
test_loader  = torch.utils.data.DataLoader(SeqDataset(X_te, y_te), batch_size=BATCH_SIZE)

# ---------------------------
# Mixup
# ---------------------------
def mixup_data(x, y, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    mixed_x = lam * x + (1 - lam) * x[idx].to(x.device)
    y_a, y_b = y, y[idx].to(y.device)
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# ---------------------------
# Model Components
# ---------------------------
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=8):
        super().__init__()
        r = max(1, channel // reduction)
        self.fc1 = nn.Linear(channel, r)
        self.fc2 = nn.Linear(r, channel)
    def forward(self, x):
        w = x.mean(dim=2)
        w = F.relu(self.fc1(w))
        w = torch.sigmoid(self.fc2(w)).unsqueeze(2)
        return x * w

class TransformerEncoder(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2, dim_feedforward=256, dropout=0.3):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
    def forward(self, x):
        return self.transformer(x)  # (B, T, D)

class MST_Transformer(nn.Module):
    def __init__(self, input_dim, seq_len, num_classes,
                 cnn_channels=64, embed_dim=128, num_heads=4, dropout=0.3):
        super().__init__()
        # CNN stack
        self.conv1 = nn.Conv1d(input_dim, cnn_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels//2, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(cnn_channels//2)
        self.relu = nn.ReLU()

        # SE
        self.se = SEBlock(cnn_channels//2)

        # Transformer path
        self.proj = nn.Linear(input_dim, embed_dim)
        self.transformer = TransformerEncoder(d_model=embed_dim, nhead=num_heads, num_layers=2)

        # Fused dim
        fused_dim = embed_dim + (cnn_channels//2)
        self.gate = nn.Linear(fused_dim, fused_dim)
        self.attn = nn.MultiheadAttention(embed_dim=fused_dim, num_heads=num_heads, batch_first=True)
        self.ln = nn.LayerNorm(fused_dim)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(fused_dim, num_classes)

    def forward(self, x):
        B, T, F_in = x.shape
        # CNN
        c = x.permute(0,2,1)
        c = self.relu(self.bn1(self.conv1(c)))
        c = self.relu(self.bn2(self.conv2(c)))
        c = self.se(c)
        f_cnn = c.mean(dim=2)
        # Transformer
        t_in = self.proj(x)
        t_out = self.transformer(t_in)
        f_trans = t_out.mean(dim=1)
        # concat
        f_cat = torch.cat([f_cnn, f_trans], dim=1)
        gate = torch.sigmoid(self.gate(f_cat))
        f_gate = f_cat * gate
        q = f_gate.unsqueeze(1)
        attn_out, _ = self.attn(q, q, q)
        attn_out = self.ln(attn_out.squeeze(1))
        out = self.dropout(attn_out)
        logits = self.fc(out)
        return logits

# ---------------------------
# Train + Eval
# ---------------------------
input_dim = X_seq.shape[2]
num_classes = n_classes
model = MST_Transformer(input_dim, SEQ_LEN, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

best_val, best_path = -1, "best_MST_Transformer.pth"
t0 = time.time()
for epoch in range(1, EPOCHS+1):
    model.train()
    total_loss, total, correct = 0, 0, 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        X_mix, y_a, y_b, lam = mixup_data(Xb, yb)
        optimizer.zero_grad()
        out = model(X_mix)
        loss = mixup_criterion(criterion, out, y_a, y_b, lam)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * Xb.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == y_a).sum().item()
        total += Xb.size(0)
    # Val
    model.eval()
    val_preds, val_true = [], []
    with torch.no_grad():
        for Xv, yv in val_loader:
            Xv, yv = Xv.to(device), yv.to(device)
            pv = model(Xv).argmax(dim=1)
            val_preds += pv.cpu().tolist()
            val_true += yv.cpu().tolist()
    val_acc = accuracy_score(val_true, val_preds)
    print(f"Epoch {epoch}/{EPOCHS} Loss {total_loss/total:.4f} ValAcc {val_acc:.4f}")
    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), best_path)
        print("  >> Saved best model")

train_time = time.time() - t0
print(f"Training done in {train_time:.1f}s | best val={best_val:.4f}")

# ---------------------------
# Evaluate + Save Excel
# ---------------------------
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()
if torch.cuda.is_available(): torch.cuda.synchronize()
tt0 = time.time()
preds, probs, trues = [], [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        out = model(Xb)
        p = torch.softmax(out, dim=1)
        preds += out.argmax(1).cpu().tolist()
        probs += p.cpu().numpy().tolist()
        trues += yb.cpu().tolist()
if torch.cuda.is_available(): torch.cuda.synchronize()
tt1 = time.time()

total_pred_time = tt1 - tt0
per_sample_ms = (total_pred_time / len(trues)) * 1000
throughput = len(trues) / total_pred_time

preds, probs, trues = np.array(preds), np.array(probs), np.array(trues)
acc = accuracy_score(trues, preds)
f1 = f1_score(trues, preds, average='weighted')
prec = precision_score(trues, preds, average='weighted', zero_division=0)
rec = recall_score(trues, preds, average='weighted', zero_division=0)
try:
    from sklearn.preprocessing import label_binarize
    if num_classes > 2:
        y_onehot = label_binarize(trues, classes=np.arange(num_classes))
        roc = roc_auc_score(y_onehot, probs, average='macro', multi_class='ovr')
    else:
        roc = roc_auc_score(trues, probs[:,1])
except:
    roc = 0.0

cm = confusion_matrix(trues, preds)
cr = classification_report(trues, preds, target_names=[str(c) for c in class_names], zero_division=0)

print("\nTest Results:")
print(f"Accuracy={acc:.4f}, F1={f1:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, ROC={roc:.4f}")
print(f"Prediction Time={per_sample_ms:.3f} ms | Throughput={throughput:.1f}/s")

xls_name = "MST_Transformer_PAMAP2_Results.xlsx"
summary = {
    "Accuracy": [acc], "F1_Score": [f1], "Precision": [prec], "Recall": [rec], "ROC_AUC": [roc],
    "Parameters": [sum(p.numel() for p in model.parameters() if p.requires_grad)],
    "Model_Size_MB": [round(sum(p.numel()*p.element_size() for p in model.parameters())/1024**2, 4)],
    "Training_Time_seconds": [round(train_time, 4)],
    "Prediction_Time_ms": [round(per_sample_ms, 4)],
    "Throughput_samples_sec": [round(throughput, 2)],
    "Sequence_Length": [SEQ_LEN], "Features": [input_dim], "Test_Samples": [len(trues)]
}
df_sum = pd.DataFrame(summary)
with pd.ExcelWriter(xls_name, engine='openpyxl') as writer:
    df_sum.to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(cm).to_excel(writer, sheet_name='Confusion_Matrix', index=False)
    cr_dict = classification_report(trues, preds, output_dict=True, zero_division=0)
    pd.DataFrame(cr_dict).transpose().to_excel(writer, sheet_name='Classification_Report')
print("✅ Results saved to", xls_name)


Using device: cuda
1) Loading and preprocessing...
Loading PAMAP2 dataset from: PAMAP2_Dataset
Loaded: (2724953, 40) labels: (2724953,)
Sequences: (136243, 100, 40) Labels: [ 9625  9259  9495 11941  4911  8229  9404  4182 15497  2726  5862  5244
  8768 11936  4994  9359  2343  2468]
Epoch 1/20 Loss 0.8667 ValAcc 0.9789
  >> Saved best model
Epoch 2/20 Loss 0.6916 ValAcc 0.9856
  >> Saved best model
Epoch 3/20 Loss 0.6496 ValAcc 0.9888
  >> Saved best model
Epoch 4/20 Loss 0.6228 ValAcc 0.9889
  >> Saved best model
Epoch 5/20 Loss 0.6118 ValAcc 0.9925
  >> Saved best model
Epoch 6/20 Loss 0.6027 ValAcc 0.9919
Epoch 7/20 Loss 0.5964 ValAcc 0.9912
Epoch 8/20 Loss 0.5803 ValAcc 0.9929
  >> Saved best model
Epoch 9/20 Loss 0.5642 ValAcc 0.9948
  >> Saved best model
Epoch 10/20 Loss 0.5953 ValAcc 0.9945
Epoch 11/20 Loss 0.5640 ValAcc 0.9945
Epoch 12/20 Loss 0.5601 ValAcc 0.9931
Epoch 13/20 Loss 0.5694 ValAcc 0.9948
  >> Saved best model
Epoch 14/20 Loss 0.5396 ValAcc 0.9944
Epoch 15/20 Loss 

In [1]:
# =====================================
# MST WITH TRANSFORMER FOR PAMAP2 DATASET
# =====================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                           recall_score, roc_auc_score, mean_absolute_error,
                           mean_squared_error, r2_score, confusion_matrix,
                           classification_report)
import time
import psutil
import os
from datetime import datetime
import math

# Force CPU for stability
device = torch.device('cpu')
print("Using CPU for stability")

# =====================================
# 1. Data Loading
# =====================================
def load_pamap2_data(data_path="PAMAP2_Dataset"):
    print("Loading PAMAP2 dataset from local files...")
    all_data = []
    all_labels = []
    
    column_names = ['timestamp', 'activity_id', 'heart_rate']
    imu_columns = [
        'temperature', 'acceleration_16g_x', 'acceleration_16g_y', 'acceleration_16g_z',
        'acceleration_6g_x', 'acceleration_6g_y', 'acceleration_6g_z',
        'gyroscope_x', 'gyroscope_y', 'gyroscope_z',
        'magnetometer_x', 'magnetometer_y', 'magnetometer_z',
        'orientation_1', 'orientation_2', 'orientation_3', 'orientation_4'
    ]
    
    for position in ['hand', 'chest', 'ankle']:
        for col in imu_columns:
            column_names.append(f'{position}_{col}')
    
    # Load Protocol data
    protocol_path = os.path.join(data_path, "Protocol")
    if os.path.exists(protocol_path):
        print("Loading Protocol data...")
        for subject_file in os.listdir(protocol_path):
            if subject_file.endswith('.dat'):
                file_path = os.path.join(protocol_path, subject_file)
                try:
                    subject_data = pd.read_csv(file_path, sep=' ', header=None, names=column_names)
                    subject_data = subject_data[subject_data['activity_id'] != 0]
                    orientation_cols = [col for col in subject_data.columns if 'orientation' in col]
                    subject_data = subject_data.drop(columns=orientation_cols)
                    feature_cols = [col for col in subject_data.columns if col not in ['timestamp', 'activity_id']]
                    all_data.append(subject_data[feature_cols])
                    all_labels.append(subject_data['activity_id'])
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    
    # Load Optional data
    optional_path = os.path.join(data_path, "Optional")
    if os.path.exists(optional_path):
        print("Loading Optional data...")
        for subject_file in os.listdir(optional_path):
            if subject_file.endswith('.dat'):
                file_path = os.path.join(optional_path, subject_file)
                try:
                    subject_data = pd.read_csv(file_path, sep=' ', header=None, names=column_names)
                    subject_data = subject_data[subject_data['activity_id'] != 0]
                    orientation_cols = [col for col in subject_data.columns if 'orientation' in col]
                    subject_data = subject_data.drop(columns=orientation_cols)
                    feature_cols = [col for col in subject_data.columns if col not in ['timestamp', 'activity_id']]
                    all_data.append(subject_data[feature_cols])
                    all_labels.append(subject_data['activity_id'])
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
    
    X = pd.concat(all_data, ignore_index=True)
    y = pd.concat(all_labels, ignore_index=True)
    
    print(f"Loaded dataset shape: {X.shape}")
    return X, y

# =====================================
# 2. MST with Transformer Model Architecture
# =====================================
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff=256, dropout=0.1):
        super().__init__()
        self.self_attention = nn.MultiheadAttention(
            embed_dim=d_model, 
            num_heads=n_heads, 
            dropout=dropout, 
            batch_first=True
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # Self-attention with residual connection and layer norm
        attn_out, _ = self.self_attention(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward with residual connection and layer norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        
        return x

class MultiScaleTransformer(nn.Module):
    def __init__(self, input_dim, num_classes, seq_len=100, d_model=64, n_heads=4, n_layers=2):
        super().__init__()
        self.seq_len = seq_len
        self.d_model = d_model
        
        # Multi-scale input projections
        self.scale1_proj = nn.Linear(input_dim, d_model)  # Original scale
        self.scale2_proj = nn.Linear(input_dim, d_model)  # Downsample by 2
        self.scale3_proj = nn.Linear(input_dim, d_model)  # Downsample by 4
        
        # Positional encodings for different scales
        self.pos_encoding1 = nn.Parameter(torch.randn(1, seq_len, d_model))
        self.pos_encoding2 = nn.Parameter(torch.randn(1, seq_len//2, d_model))
        self.pos_encoding3 = nn.Parameter(torch.randn(1, seq_len//4, d_model))
        
        # Transformer layers for each scale
        self.transformer_layers1 = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff=d_model*2, dropout=0.1)
            for _ in range(n_layers)
        ])
        
        self.transformer_layers2 = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff=d_model*2, dropout=0.1)
            for _ in range(n_layers)
        ])
        
        self.transformer_layers3 = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff=d_model*2, dropout=0.1)
            for _ in range(n_layers)
        ])
        
        # Feature fusion
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 3, d_model),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        
        # Classifier (same structure as original)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
        
    def downsample(self, x, scale_factor):
        """Downsample sequence by average pooling"""
        B, T, D = x.shape
        x = x.reshape(B, T // scale_factor, scale_factor, D)
        return x.mean(dim=2)
        
    def forward(self, x):
        # x shape: [B, seq_len, input_dim]
        B, T, D = x.shape
        
        # Scale 1: Original resolution
        x1 = self.scale1_proj(x)
        x1 = x1 + self.pos_encoding1
        
        for transformer in self.transformer_layers1:
            x1 = transformer(x1)
        features1 = x1.mean(dim=1)  # Global average pooling
        
        # Scale 2: Downsample by 2
        x2 = self.downsample(x, 2)
        x2 = self.scale2_proj(x2)
        x2 = x2 + self.pos_encoding2
        
        for transformer in self.transformer_layers2:
            x2 = transformer(x2)
        features2 = x2.mean(dim=1)  # Global average pooling
        
        # Scale 3: Downsample by 4
        x3 = self.downsample(x, 4)
        x3 = self.scale3_proj(x3)
        x3 = x3 + self.pos_encoding3
        
        for transformer in self.transformer_layers3:
            x3 = transformer(x3)
        features3 = x3.mean(dim=1)  # Global average pooling
        
        # Feature fusion
        fused_features = torch.cat([features1, features2, features3], dim=1)
        fused_features = self.fusion(fused_features)
        
        # Classification
        output = self.classifier(fused_features)
        
        return output

class PAMAP2Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# =====================================
# 3. Training Function
# =====================================
def train_mst_transformer_model(model, train_loader, val_loader, epochs=10):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    train_losses = []
    val_accuracies = []
    best_val_acc = 0
    training_start = time.time()
    
    for epoch in range(epochs):
        # Training
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total_train += target.size(0)
            correct_train += (predicted == target).sum().item()
            
            if batch_idx % 100 == 0:
                print(f'  Batch {batch_idx}, Loss: {loss.item():.4f}')
        
        scheduler.step()
        
        train_acc = 100 * correct_train / total_train
        avg_loss = running_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        # Validation
        model.eval()
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for data, target in val_loader:
                output = model(data)
                _, predicted = torch.max(output.data, 1)
                total_val += target.size(0)
                correct_val += (predicted == target).sum().item()
        
        val_acc = 100 * correct_val / total_val
        val_accuracies.append(val_acc)
        
        print(f'Epoch {epoch+1}/{epochs}:')
        print(f'  Loss: {avg_loss:.4f}, Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%')
        print(f'  Learning Rate: {scheduler.get_last_lr()[0]:.6f}')
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_mst_transformer_model.pth')
            print(f'  → New best model saved! (Val Acc: {val_acc:.2f}%)')
    
    training_time = time.time() - training_start
    return train_losses, val_accuracies, training_time

# =====================================
# 4. Main Execution
# =====================================
def main():
    print("🚀 Starting MST with Transformer Model Training on PAMAP2 Dataset")
    
    # Load and preprocess data
    print("\n📊 Loading data...")
    X, y = load_pamap2_data("PAMAP2_Dataset")
    
    # Preprocessing
    X_clean = X.fillna(X.mean())
    constant_cols = X_clean.columns[X_clean.nunique() <= 1]
    if constant_cols.any():
        X_clean = X_clean.drop(columns=constant_cols)
        print(f"Removed {len(constant_cols)} constant columns")
    
    X_array = X_clean.values.astype(np.float32)
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    print(f"Classes: {np.unique(y_encoded)}")
    
    # Feature scaling
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_array)
    
    # Create sequences
    def create_sequences(X, y, seq_len=100, stride=20):
        X_seq, y_seq = [], []
        for i in range(0, len(X) - seq_len, stride):
            X_seq.append(X[i:i+seq_len])
            sequence_labels = y[i:i+seq_len]
            most_common_label = np.argmax(np.bincount(sequence_labels))
            y_seq.append(most_common_label)
        return np.array(X_seq, dtype=np.float32), np.array(y_seq, dtype=np.int64)
    
    sequence_length = 100
    stride = 20
    X_seq, y_seq = create_sequences(X_scaled, y_encoded, seq_len=sequence_length, stride=stride)
    
    print(f"Sequences created: {X_seq.shape}, Labels: {y_seq.shape}")
    
    # Split data
    X_train, X_temp, y_train, y_temp = train_test_split(X_seq, y_seq, test_size=0.3, stratify=y_seq, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)
    
    print(f"Data splits - Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    
    # Model configuration
    input_dim = X_train.shape[2]
    num_classes = len(np.unique(y_train))
    
    print(f"\n⚙️ Model Configuration:")
    print(f"Input dimension: {input_dim}")
    print(f"Sequence length: {sequence_length}")
    print(f"Number of classes: {num_classes}")
    
    # Create datasets and loaders
    train_dataset = PAMAP2Dataset(X_train, y_train)
    val_dataset = PAMAP2Dataset(X_val, y_val)
    test_dataset = PAMAP2Dataset(X_test, y_test)
    
    batch_size = 32
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # Initialize MST with Transformer model
    model = MultiScaleTransformer(
        input_dim=input_dim,
        num_classes=num_classes,
        seq_len=sequence_length,
        d_model=64,
        n_heads=4,
        n_layers=2
    )
    
    # Calculate parameters
    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    total_params = count_parameters(model)
    print(f"Model parameters: {total_params:,}")
    
    # Test forward pass
    print("\n🧪 Testing forward pass...")
    try:
        test_batch = torch.tensor(X_train[:2], dtype=torch.float32)
        with torch.no_grad():
            output = model(test_batch)
        print(f"Forward pass successful! Output shape: {output.shape}")
    except Exception as e:
        print(f"Forward pass failed: {e}")
        return
    
    # Training
    print(f"\n{'='*60}")
    print(f"Training MST with Transformer Model")
    print(f"{'='*60}")
    
    train_losses, val_accuracies, training_time = train_mst_transformer_model(model, train_loader, val_loader, epochs=10)
    
    # =====================================
    # 5. Evaluation and Excel Export
    # =====================================
    print(f"\n{'='*60}")
    print(f"Evaluating MST with Transformer Model")
    print(f"{'='*60}")
    
    # Load best model
    try:
        model.load_state_dict(torch.load('best_mst_transformer_model.pth'))
        print("Loaded best model for evaluation")
    except:
        print("Using final model for evaluation")
    
    model.eval()
    
    # Evaluation on test set
    all_predictions, all_probabilities, all_targets = [], [], []
    prediction_start = time.time()
    
    with torch.no_grad():
        for data, target in test_loader:
            output = model(data)
            probabilities = torch.softmax(output, dim=1)
            _, predictions = torch.max(output, 1)
            all_predictions.extend(predictions.numpy())
            all_probabilities.extend(probabilities.numpy())
            all_targets.extend(target.numpy())
    
    total_prediction_time = (time.time() - prediction_start) * 1000
    prediction_time_per_sample = total_prediction_time / len(all_targets)
    
    all_predictions = np.array(all_predictions)
    all_probabilities = np.array(all_probabilities)
    all_targets = np.array(all_targets)
    
    # Calculate metrics
    accuracy = accuracy_score(all_targets, all_predictions)
    f1 = f1_score(all_targets, all_predictions, average='weighted')
    precision = precision_score(all_targets, all_predictions, average='weighted')
    recall = recall_score(all_targets, all_predictions, average='weighted')
    
    try:
        roc_auc = roc_auc_score(all_targets, all_probabilities, multi_class='ovr', average='weighted')
    except:
        roc_auc = 0.0
    
    mae = mean_absolute_error(all_targets, all_predictions)
    mse = mean_squared_error(all_targets, all_predictions)
    r2 = r2_score(all_targets, all_predictions)
    
    # Throughput
    test_batch = torch.tensor(X_test[:batch_size], dtype=torch.float32)
    start_time = time.time()
    with torch.no_grad():
        _ = model(test_batch)
    batch_time = (time.time() - start_time) * 1000
    throughput = (batch_size / batch_time) * 1000
    
    # Memory usage
    process = psutil.Process(os.getpid())
    memory_usage = process.memory_info().rss / (1024 ** 2)
    
    # FLOPs estimation for MST with Transformer
    def estimate_mst_transformer_flops(input_shape):
        batch_size, seq_len, input_dim = input_shape
        d_model = 64
        n_heads = 4
        n_layers = 2
        
        # Scale 1 FLOPs (original resolution)
        # Input projection
        scale1_proj_flops = 2 * input_dim * d_model * seq_len
        
        # Transformer layers FLOPs
        # Self-attention: 4 * d_model^2 * seq_len + 2 * d_model * seq_len^2
        attention_flops_per_layer = (4 * d_model * d_model * seq_len + 
                                   2 * d_model * seq_len * seq_len)
        # FFN FLOPs: 2 * d_model * (2*d_model) * seq_len
        ffn_flops_per_layer = 2 * d_model * (2 * d_model) * seq_len
        scale1_transformer_flops = n_layers * (attention_flops_per_layer + ffn_flops_per_layer)
        
        # Scale 2 FLOPs (downsample by 2)
        scale2_seq_len = seq_len // 2
        scale2_proj_flops = 2 * input_dim * d_model * scale2_seq_len
        attention_flops_scale2 = (4 * d_model * d_model * scale2_seq_len + 
                                2 * d_model * scale2_seq_len * scale2_seq_len)
        ffn_flops_scale2 = 2 * d_model * (2 * d_model) * scale2_seq_len
        scale2_transformer_flops = n_layers * (attention_flops_scale2 + ffn_flops_scale2)
        
        # Scale 3 FLOPs (downsample by 4)
        scale3_seq_len = seq_len // 4
        scale3_proj_flops = 2 * input_dim * d_model * scale3_seq_len
        attention_flops_scale3 = (4 * d_model * d_model * scale3_seq_len + 
                                2 * d_model * scale3_seq_len * scale3_seq_len)
        ffn_flops_scale3 = 2 * d_model * (2 * d_model) * scale3_seq_len
        scale3_transformer_flops = n_layers * (attention_flops_scale3 + ffn_flops_scale3)
        
        # Fusion and classifier FLOPs
        fusion_flops = 2 * (d_model * 3) * d_model
        classifier_flops = 2 * d_model * 128 + 2 * 128 * 64 + 2 * 64 * num_classes
        
        total_flops = (scale1_proj_flops + scale1_transformer_flops +
                      scale2_proj_flops + scale2_transformer_flops +
                      scale3_proj_flops + scale3_transformer_flops +
                      fusion_flops + classifier_flops)
        
        return total_flops
    
    flops = estimate_mst_transformer_flops((1, sequence_length, input_dim))
    
    # Calculate latency
    latency_per_sample = 1 / (throughput / 1000) * 1000 if throughput > 0 else 0
    
    # =====================================
    # 6. Save Results to Excel
    # =====================================
    print("\n💾 Saving results to Excel...")
    
    # Create comprehensive results
    results_data = {
        'Metric_Type': ['Performance', 'Performance', 'Performance', 'Performance', 'Performance', 
                       'Performance', 'Performance', 'Performance', 'Model', 'Model', 'Model', 
                       'Model', 'Model', 'Model', 'Model', 'Model', 'Model', 'Model', 'Dataset'],
        'Metric_Name': ['Accuracy', 'F1_Score', 'Precision', 'Recall', 'ROC_AUC', 'MAE', 'MSE', 'R2_Score',
                       'Parameters', 'Model_Size_MB', 'Training_Time_seconds', 'Prediction_Time_ms', 
                       'Throughput_samples_sec', 'Latency_ms', 'Sequence_Length', 'Features', 'FLOPs', 
                       'Memory_Usage_MB', 'Test_Samples'],
        'Value': [accuracy, f1, precision, recall, roc_auc, mae, mse, r2,
                 total_params, total_params * 4 / (1024**2), training_time, 
                 prediction_time_per_sample, throughput, latency_per_sample, sequence_length, 
                 input_dim, flops, memory_usage, len(X_test)],
        'Description': [
            'Classification accuracy', 'Weighted F1-score', 'Weighted precision', 'Weighted recall',
            'Area under ROC curve', 'Mean Absolute Error', 'Mean Squared Error', 'R-squared score',
            'Total trainable parameters', 'Model size in megabytes', 'Time taken for training', 
            'Prediction time per sample', 'Samples processed per second', 'Latency per prediction',
            'Sequence length for time series', 'Number of input features', 'Floating point operations',
            'Memory usage during inference', 'Number of test samples'
        ]
    }
    
    results_df = pd.DataFrame(results_data)
    
    # Create detailed classification report
    classification_rep = classification_report(all_targets, all_predictions, output_dict=True)
    class_report_df = pd.DataFrame(classification_rep).transpose()
    
    # Create confusion matrix data
    cm = confusion_matrix(all_targets, all_predictions)
    cm_df = pd.DataFrame(cm, index=[f'True_{i}' for i in range(cm.shape[0])], 
                        columns=[f'Pred_{i}' for i in range(cm.shape[1])])
    
    # Create training history
    training_history = pd.DataFrame({
        'Epoch': range(1, len(train_losses) + 1),
        'Training_Loss': train_losses,
        'Validation_Accuracy': val_accuracies
    })
    
    # Save to Excel with multiple sheets
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f'MST_Transformer_PAMAP2_Results_{timestamp}.xlsx'
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        results_df.to_excel(writer, sheet_name='Summary_Metrics', index=False)
        class_report_df.to_excel(writer, sheet_name='Detailed_Classification_Report')
        cm_df.to_excel(writer, sheet_name='Confusion_Matrix')
        training_history.to_excel(writer, sheet_name='Training_History', index=False)
    
    # =====================================
    # 7. Display Final Results
    # =====================================
    print(f"\n{'='*80}")
    print(f"🎯 MST WITH TRANSFORMER MODEL RESULTS - PAMAP2 DATASET")
    print(f"{'='*80}")
    
    print(f"\n📊 PERFORMANCE METRICS:")
    print(f"Accuracy:       {accuracy:.4f}")
    print(f"F1-Score:       {f1:.4f}")
    print(f"Precision:      {precision:.4f}")
    print(f"Recall:         {recall:.4f}")
    print(f"ROC-AUC:        {roc_auc:.4f}")
    print(f"MAE:            {mae:.4f}")
    print(f"MSE:            {mse:.4f}")
    print(f"R² Score:       {r2:.4f}")
    
    print(f"\n⚡ EFFICIENCY METRICS:")
    print(f"Parameters:     {total_params:,}")
    print(f"Model Size:     {total_params * 4 / (1024**2):.2f} MB")
    print(f"Training Time:  {training_time:.2f} seconds")
    print(f"Prediction Time:{prediction_time_per_sample:.4f} ms")
    print(f"Throughput:     {throughput:.2f} samples/sec")
    print(f"Latency:        {latency_per_sample:.4f} ms")
    print(f"Sequence Length:{sequence_length}")
    print(f"Features:       {input_dim}")
    print(f"FLOPs:          {flops:,}")
    print(f"Memory Usage:   {memory_usage:.2f} MB")
    
    print(f"\n🔧 MODEL ARCHITECTURE:")
    print(f"Multi-Scale:    3 scales (100, 50, 25 time steps)")
    print(f"Transformer Layers: 2 per scale")
    print(f"Number of Heads: 4")
    print(f"Hidden Dimension: 64")
    print(f"Feature Fusion: Concatenation + Linear")
    
    print(f"\n💾 RESULTS SAVED TO: {filename}")
    print(f"\n✅ MST with Transformer Model Training and Evaluation Completed!")

if __name__ == "__main__":
    main()

Using CPU for stability
🚀 Starting MST with Transformer Model Training on PAMAP2 Dataset

📊 Loading data...
Loading PAMAP2 dataset from local files...
Loading Protocol data...
Loading Optional data...
Loaded dataset shape: (2724953, 40)
Classes: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17]
Sequences created: (136243, 100, 40), Labels: (136243,)
Data splits - Train: (95370, 100, 40), Val: (20436, 100, 40), Test: (20437, 100, 40)

⚙️ Model Configuration:
Input dimension: 40
Sequence length: 100
Number of classes: 18
Model parameters: 250,002

🧪 Testing forward pass...
Forward pass successful! Output shape: torch.Size([2, 18])

Training MST with Transformer Model
  Batch 0, Loss: 2.9027
  Batch 100, Loss: 2.0202
  Batch 200, Loss: 1.3669
  Batch 300, Loss: 1.1923
  Batch 400, Loss: 0.6394
  Batch 500, Loss: 0.5950
  Batch 600, Loss: 0.5962
  Batch 700, Loss: 0.4335
  Batch 800, Loss: 0.8682
  Batch 900, Loss: 0.5873
  Batch 1000, Loss: 0.4211
  Batch 1100, Loss: 0.5344
  Batch 